In [ ]:
# ============================================================
# AI-BASED TRAFFIC CONDITION CLASSIFICATION SYSTEM
# ONE COMPLETE PROJECT PROGRAM — RUN THIS CELL ONCE
# ============================================================

# SECTION 0: IMPORT LIBRARIES AND UPLOAD DATASETS
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)

warnings.filterwarnings("ignore")

print("=" * 65)
print("AI-BASED TRAFFIC CONDITION CLASSIFICATION SYSTEM")
print("=" * 65)
print("Upload BOTH datasets when the upload window appears:")
print("1. traffic_weather_full2020.csv")
print("2. Metro_Interstate_Traffic_Volume.csv.gz")

uploaded = files.upload()

dataset1_file = next(
    (name for name in uploaded
     if "traffic_weather_full2020" in name.lower() and name.lower().endswith(".csv")),
    None
)
dataset2_file = next(
    (name for name in uploaded if name.lower().endswith(".csv.gz")),
    None
)

if dataset1_file is None or dataset2_file is None:
    raise FileNotFoundError(
        "Both files are required: traffic_weather_full2020.csv "
        "and Metro_Interstate_Traffic_Volume.csv.gz. "
        "Run this cell again and upload both files."
    )

# SECTION 1: LOAD DATASET 1
df = pd.read_csv(dataset1_file)
print("\\nSECTION 1 — DATASET 1 LOADED")
print("File:", dataset1_file)
print("Shape:", df.shape)
display(df.head())



# ==========================================================
# SECTION 2 — DATA QUALITY CHECK
# ==========================================================


# Task 2: Data Quality Check

print("Missing Values:")
print(df.isnull().sum())

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nDataset Information:")
df.info()



# ==========================================================
# SECTION 3 — TRAFFIC FLOW ANALYSIS
# ==========================================================


# Task 3: Traffic Flow Analysis

print("TRAFFIC FLOW STATISTICS")
print("-----------------------------")

print(df["Flow"].describe())

print("\nMinimum Flow:", df["Flow"].min())
print("Maximum Flow:", df["Flow"].max())
print("Average Flow:", df["Flow"].mean())
print("Median Flow:", df["Flow"].median())

print("\nUnique Flow Values:", df["Flow"].nunique())



# ==========================================================
# SECTION 4 — TRAFFIC CLASSIFICATION
# ==========================================================


# Task 4: Traffic Classification

def classify_traffic(flow):
    if flow <= 237:
        return "Low"
    elif flow <= 568:
        return "Medium"
    else:
        return "High"

df["Traffic_Category"] = df["Flow"].apply(classify_traffic)

print("Traffic Categories Created Successfully!")

display(df[["Flow", "Traffic_Category"]].head(10))



# ==========================================================
# SECTION 5 — CLASS DISTRIBUTION
# ==========================================================


# Check Traffic Class Balance

print("Traffic Category Distribution:")
print(df["Traffic_Category"].value_counts())

print("\nTraffic Category Percentages:")
print(
    (df["Traffic_Category"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nTotal Records:", len(df))



# ==========================================================
# SECTION 6 — VISUALIZATION 1: TRAFFIC CATEGORY DISTRIBUTION
# ==========================================================


import matplotlib.pyplot as plt

# Traffic Category Distribution

category_counts = df["Traffic_Category"].value_counts()

plt.figure(figsize=(8, 5))

category_counts.plot(kind="bar")

plt.title("Traffic Category Distribution")
plt.xlabel("Traffic Category")
plt.ylabel("Number of Records")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()



# ==========================================================
# SECTION 7 — VISUALIZATION 2: TRAFFIC FLOW DISTRIBUTION
# ==========================================================


# Visualization 2: Traffic Flow Distribution

plt.figure(figsize=(8, 5))

plt.hist(df["Flow"], bins=30, edgecolor="black")

plt.title("Traffic Flow Distribution")
plt.xlabel("Traffic Flow")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()



# ==========================================================
# SECTION 8 — VISUALIZATION 3: TRAFFIC BY WEATHER CONDITION
# ==========================================================


# Visualization 3: Traffic Flow by Weather Condition

weather_traffic = df.groupby("Condition")["Flow"].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))

weather_traffic.plot(kind="bar", edgecolor="black")

plt.title("Average Traffic Flow by Weather Condition")
plt.xlabel("Weather Condition")
plt.ylabel("Average Traffic Flow")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()



# ==========================================================
# SECTION 9 — VISUALIZATION 4: TRAFFIC BY HOUR
# ==========================================================


# Visualization 4: Traffic Flow by Hour

df["Hour_Number"] = pd.to_datetime(
    df["5 Minutes"]
).dt.hour

hourly_traffic = df.groupby("Hour_Number")["Flow"].mean()

plt.figure(figsize=(10, 5))

plt.plot(hourly_traffic.index, hourly_traffic.values, marker="o")

plt.title("Average Traffic Flow by Hour of the Day")
plt.xlabel("Hour of the Day")
plt.ylabel("Average Traffic Flow")
plt.xticks(range(0, 24))
plt.grid(True)

plt.tight_layout()
plt.show()



# ==========================================================
# SECTION 10 — DATA PREPROCESSING INSPECTION
# ==========================================================


# Data Preprocessing: Initial Inspection

print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nUNIQUE VALUES IN IMPORTANT COLUMNS")

for column in ["Temperature", "Dew Point", "Wind",
               "Wind Speed", "Wind Gust", "Condition"]:
    print(f"\n{column}:")
    print(df[column].dropna().unique()[:10])



# ==========================================================
# SECTION 11 — DATA CLEANING
# ==========================================================


# Data Cleaning and Preprocessing

import pandas as pd
import numpy as np

# Create a cleaned copy
df_clean = df.copy()

# Convert Temperature and Dew Point into numeric values
for column in ["Temperature", "Dew Point"]:
    df_clean[column] = pd.to_numeric(
        df_clean[column].astype(str).str.replace(
            r"[^0-9.-]", "", regex=True
        ),
        errors="coerce"
    )

# Convert Wind Speed and Wind Gust into numeric values
for column in ["Wind Speed", "Wind Gust"]:
    df_clean[column] = pd.to_numeric(
        df_clean[column], errors="coerce"
    )

# Handle missing numerical values using median
numeric_columns = [
    "Temperature", "Dew Point",
    "Wind Speed", "Wind Gust"
]

for column in numeric_columns:
    df_clean[column] = df_clean[column].fillna(
        df_clean[column].median()
    )

# Handle missing categorical values
df_clean["Wind"] = df_clean["Wind"].fillna("Unknown")
df_clean["Condition"] = df_clean["Condition"].fillna("Unknown")

# Display cleaned dataset information
print("DATA CLEANING COMPLETED")

print("\nUpdated Data Types:")
print(df_clean.dtypes)

print("\nMissing Values After Cleaning:")
print(df_clean.isnull().sum())

print("\nCleaned Dataset Shape:")
print(df_clean.shape)

display(df_clean.head())



# ==========================================================
# SECTION 12 — FEATURE SELECTION
# ==========================================================


# Step 1: Feature Selection

# Select input features
features = [
    "(mph)",
    "week_day",
    "holidays",
    "Temperature",
    "Dew Point",
    "Humidity",
    "Wind",
    "Wind Speed",
    "Wind Gust",
    "Pressure",
    "Precip.",
    "Condition",
    "Hour_Number"
]

# Input variables
X = df_clean[features]

# Target variable
y = df_clean["Traffic_Category"]

print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nInput Features:")
print(X.head())

print("\nTarget Categories:")
print(y.head())



# ==========================================================
# SECTION 13 — MACHINE LEARNING PREPROCESSING AND RANDOM SPLIT
# ==========================================================


# STEP 1: Machine Learning Preprocessing

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Work from the original dataset
model_df = df.copy()

# Convert text-based numerical columns
for col in ["Temperature", "Dew Point", "Wind Speed", "Wind Gust"]:
    model_df[col] = pd.to_numeric(
        model_df[col].astype(str).str.replace(
            r"[^0-9.-]", "", regex=True
        ),
        errors="coerce"
    )

# Define input features
features = [
    "(mph)", "week_day", "holidays",
    "Temperature", "Dew Point", "Humidity",
    "Wind", "Wind Speed", "Wind Gust",
    "Pressure", "Precip.", "Condition", "Hour_Number"
]

X = model_df[features]
y = model_df["Traffic_Category"]

# Separate numerical and categorical features
numeric_features = [
    "(mph)", "week_day", "holidays",
    "Temperature", "Dew Point", "Humidity",
    "Wind Speed", "Wind Gust", "Pressure",
    "Precip.", "Hour_Number"
]

categorical_features = ["Wind", "Condition"]

# Prepare numerical data
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Prepare categorical data
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant",
                              fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Split training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("\nPreprocessing pipeline created successfully!")



# ==========================================================
# SECTION 14 — EXPERIMENT 1: DECISION TREE
# ==========================================================


# Decision Tree Classifier - Baseline Model

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

decision_tree = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Train model
decision_tree.fit(X_train, y_train)

# Make predictions
dt_predictions = decision_tree.predict(X_test)

# Evaluate model
print("DECISION TREE RESULTS")
print("-----------------------------")

print("Accuracy:",
      accuracy_score(y_test, dt_predictions))

print("\nClassification Report:")
print(classification_report(y_test, dt_predictions))



# ==========================================================
# SECTION 15 — EXPERIMENT 2: RANDOM FOREST
# ==========================================================


# Model 2: Random Forest Classifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

random_forest = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train model
random_forest.fit(X_train, y_train)

# Make predictions
rf_predictions = random_forest.predict(X_test)

# Evaluate model
print("RANDOM FOREST RESULTS")
print("-----------------------------")

print("Accuracy:",
      accuracy_score(y_test, rf_predictions))

print("\nClassification Report:")
print(classification_report(y_test, rf_predictions))



# ==========================================================
# SECTION 16 — CONFUSION MATRICES: DECISION TREE AND RANDOM FOREST
# ==========================================================

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, dt_predictions,
    ax=axes[0],
    cmap="Blues"
)
axes[0].set_title("Decision Tree Confusion Matrix")

ConfusionMatrixDisplay.from_predictions(
    y_test, rf_predictions,
    ax=axes[1],
    cmap="Greens"
)
axes[1].set_title("Random Forest Confusion Matrix")

plt.tight_layout()
plt.show()



# ==========================================================
# SECTION 17 — EXPERIMENT 3: TUNED RANDOM FOREST
# ==========================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

tuned_random_forest = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

tuned_random_forest.fit(X_train, y_train)

tuned_predictions = tuned_random_forest.predict(X_test)

print("TUNED RANDOM FOREST RESULTS")
print("-----------------------------")
print("Accuracy:", accuracy_score(y_test, tuned_predictions))
print("\nClassification Report:")
print(classification_report(y_test, tuned_predictions))



# ==========================================================
# SECTION 18 — TUNED RANDOM FOREST CONFUSION MATRIX
# ==========================================================

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    tuned_predictions,
    cmap="Purples"
)

plt.title("Tuned Random Forest Confusion Matrix")
plt.show()



# ==========================================================
# SECTION 19 — EDGE-CASE TESTING
# ==========================================================

# Create three test cases using existing feature columns
edge_cases = X_test.iloc[[0, 1, 2]].copy()

# Test 1: Missing values
edge_cases.iloc[0, edge_cases.columns.get_loc("Temperature")] = float("nan")
edge_cases.iloc[0, edge_cases.columns.get_loc("Wind")] = None

# Test 2: New/unseen categories
edge_cases.iloc[1, edge_cases.columns.get_loc("Condition")] = "Unknown Weather"
edge_cases.iloc[1, edge_cases.columns.get_loc("Wind")] = "Unknown Wind"

# Test 3: Extreme numerical values
edge_cases.iloc[2, edge_cases.columns.get_loc("(mph)")] = 1000
edge_cases.iloc[2, edge_cases.columns.get_loc("Temperature")] = -100

# Predict using tuned Random Forest
edge_predictions = tuned_random_forest.predict(edge_cases)

print("EDGE CASE TEST RESULTS")
print("----------------------")

for i, prediction in enumerate(edge_predictions, start=1):
    print(f"Test {i}: Predicted Traffic = {prediction}")

print("\nAll three test cases processed successfully.")



# ==========================================================
# SECTION 20 — LOAD AND EXPLORE DATASET 2
# ==========================================================

# Load the second dataset from the file uploaded at the start
df2 = pd.read_csv(dataset2_file, compression="gzip")

print("FIRST FIVE RECORDS")
display(df2.head())

print("\\nDATASET SHAPE:", df2.shape)
print("\\nCOLUMN NAMES:")
print(df2.columns.tolist())

print("\\nDATASET INFORMATION:")
df2.info()



# ==========================================================
# SECTION 21 — DATASET 2 QUALITY AUDIT
# ==========================================================

print("MISSING VALUES")
print(df2.isnull().sum())

print("\nTOTAL MISSING VALUES:")
print(df2.isnull().sum().sum())

print("\nDUPLICATE ROWS:")
print(df2.duplicated().sum())

print("\nNUMERICAL SUMMARY:")
display(df2.describe())

print("\nTRAFFIC VOLUME DISTRIBUTION:")
print(df2["traffic_volume"].describe())

print("\nTRAFFIC VOLUME CATEGORIES:")
print(df2["traffic_volume"].value_counts().head(10))



# ==========================================================
# SECTION 22 — INVESTIGATE UNUSUAL DATASET 2 VALUES
# ==========================================================

print("HOLIDAY VALUE COUNTS:")
print(df2["holiday"].value_counts(dropna=False))

print("\nTEMPERATURE VALUES BELOW 200:")
print(df2[df2["temp"] < 200][["temp", "date_time"]].head(10))

print("\nRAINFALL VALUES ABOVE 100:")
print(df2[df2["rain_1h"] > 100][["rain_1h", "date_time"]].head(10))

print("\nTRAFFIC VOLUME ZERO RECORDS:")
print((df2["traffic_volume"] == 0).sum())

print("\nDATE RANGE:")
print(df2["date_time"].min(), "to", df2["date_time"].max())



# ==========================================================
# SECTION 23 — INSPECT DUPLICATES AND SUSPICIOUS RECORDS
# ==========================================================

print("DUPLICATE RECORDS:")
display(df2[df2.duplicated(keep=False)].sort_values("date_time").head(20))

print("\nEXTREME RAINFALL RECORD:")
display(df2[df2["rain_1h"] > 100])

print("\nZERO TRAFFIC RECORDS:")
display(df2[df2["traffic_volume"] == 0])

print("\nZERO TEMPERATURE COUNT:")
print((df2["temp"] == 0).sum())

print("\nRAINFALL DISTRIBUTION:")
print(df2["rain_1h"].describe(percentiles=[0.90, 0.95, 0.99]))



# ==========================================================
# SECTION 24 — CLEAN DATASET 2
# ==========================================================

# Create a separate cleaned copy
df2_clean = df2.copy()

# Remove exact duplicate records
df2_clean = df2_clean.drop_duplicates()

# Convert date_time to datetime
df2_clean["date_time"] = pd.to_datetime(
    df2_clean["date_time"]
)

# Replace missing holiday values
df2_clean["holiday"] = df2_clean["holiday"].fillna("None")

# Flag suspicious rainfall values for investigation
df2_clean["rainfall_suspicious"] = (
    df2_clean["rain_1h"] > 100
)

print("ORIGINAL SHAPE:", df2.shape)
print("CLEANED SHAPE:", df2_clean.shape)

print("\nREMAINING MISSING VALUES:")
print(df2_clean.isnull().sum())

print("\nSUSPICIOUS RAINFALL RECORDS:")
display(
    df2_clean[df2_clean["rainfall_suspicious"]]
)



# ==========================================================
# SECTION 25 — CREATE DATASET 2 TRAFFIC CATEGORIES
# ==========================================================

# Calculate traffic volume thresholds
low_limit = df2_clean["traffic_volume"].quantile(0.25)
high_limit = df2_clean["traffic_volume"].quantile(0.75)

# Create traffic categories
def classify_traffic_volume(volume):
    if volume <= low_limit:
        return "Low"
    elif volume <= high_limit:
        return "Medium"
    else:
        return "High"

df2_clean["Traffic_Category"] = (
    df2_clean["traffic_volume"].apply(classify_traffic_volume)
)

print("LOW LIMIT:", low_limit)
print("HIGH LIMIT:", high_limit)

print("\nTRAFFIC CATEGORY COUNTS:")
print(df2_clean["Traffic_Category"].value_counts())

print("\nTRAFFIC CATEGORY PERCENTAGES:")
print(
    df2_clean["Traffic_Category"]
    .value_counts(normalize=True) * 100
)



# ==========================================================
# SECTION 26 — COMPARE DATASET CATEGORY DISTRIBUTIONS
# ==========================================================

import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    "Dataset 1": df_clean["Traffic_Category"].value_counts(normalize=True) * 100,
    "Dataset 2": df2_clean["Traffic_Category"].value_counts(normalize=True) * 100
})

comparison = comparison.reindex(["Low", "Medium", "High"])

comparison.plot(kind="bar", figsize=(9, 5))

plt.title("Traffic Category Distribution: Dataset 1 vs Dataset 2")
plt.xlabel("Traffic Category")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)
plt.legend(title="Dataset")
plt.tight_layout()
plt.show()



# ==========================================================
# SECTION 27 — CHRONOLOGICAL VALIDATION
# ==========================================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create a copy of Dataset 1
time_df = df_clean.copy()

# Convert date/time column
time_df["timestamp"] = pd.to_datetime(
    time_df["5 Minutes"],
    errors="coerce"
)

# Sort records chronologically
time_df = time_df.dropna(subset=["timestamp"])
time_df = time_df.sort_values("timestamp").reset_index(drop=True)

# Select features and target
X_time = time_df[features]
y_time = time_df["Traffic_Category"]

# Split: earlier 80% for training, later 20% for testing
split_index = int(len(time_df) * 0.80)

X_train_time = X_time.iloc[:split_index]
X_test_time = X_time.iloc[split_index:]

y_train_time = y_time.iloc[:split_index]
y_test_time = y_time.iloc[split_index:]

# Train Random Forest using the same tuned settings
time_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

time_model.fit(X_train_time, y_train_time)

# Evaluate on later records
time_predictions = time_model.predict(X_test_time)

print("CHRONOLOGICAL VALIDATION RESULTS")
print("--------------------------------")
print("Training records:", len(X_train_time))
print("Testing records:", len(X_test_time))
print("Training period:",
      time_df["timestamp"].iloc[0],
      "to",
      time_df["timestamp"].iloc[split_index - 1])
print("Testing period:",
      time_df["timestamp"].iloc[split_index],
      "to",
      time_df["timestamp"].iloc[-1])

print("\nAccuracy:", accuracy_score(y_test_time, time_predictions))
print("\nClassification Report:")
print(classification_report(y_test_time, time_predictions))



# ==========================================================
# SECTION 28 — CHRONOLOGICAL CONFUSION MATRIX
# ==========================================================

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test_time,
    time_predictions,
    cmap="Oranges"
)

plt.title("Chronological Validation Confusion Matrix")
plt.show()



# ==========================================================
# SECTION 29 — EXPERIMENT 4: BALANCED RANDOM FOREST
# ==========================================================

# Experiment 4: Balanced Random Forest
balanced_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Train on earlier records
balanced_model.fit(X_train_time, y_train_time)

# Predict later records
balanced_predictions = balanced_model.predict(X_test_time)

print("BALANCED RANDOM FOREST RESULTS")
print("--------------------------------")
print("Accuracy:",
      accuracy_score(y_test_time, balanced_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test_time,
    balanced_predictions
))



# ==========================================================
# SECTION 30 — BALANCED RANDOM FOREST CONFUSION MATRIX
# ==========================================================

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test_time,
    balanced_predictions,
    cmap="Greens"
)

plt.title("Balanced Random Forest Confusion Matrix")
plt.show()



# ==========================================================
# SECTION 31 — COMPARE CHRONOLOGICAL MODELS
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

models = {
    "Chronological Random Forest": time_predictions,
    "Balanced Random Forest": balanced_predictions
}

for name, predictions in models.items():
    print("\n", name)
    print("-" * 40)
    print("Accuracy:", accuracy_score(y_test_time, predictions))
    print("Macro Precision:", precision_score(
        y_test_time, predictions, average="macro"
    ))
    print("Macro Recall:", recall_score(
        y_test_time, predictions, average="macro"
    ))
    print("Macro F1:", f1_score(
        y_test_time, predictions, average="macro"
    ))



# ==========================================================
# SECTION 32 — FEATURE IMPORTANCE ANALYSIS
# ==========================================================

import pandas as pd
import matplotlib.pyplot as plt

# Get trained Random Forest classifier
rf_classifier = balanced_model.named_steps["classifier"]

# Get transformed feature names
feature_names = (
    balanced_model.named_steps["preprocessing"]
    .get_feature_names_out()
)

# Create feature importance table
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_classifier.feature_importances_
})

# Display top 15 features
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("TOP 15 IMPORTANT FEATURES")
display(importance_df.head(15))

# Plot feature importance
importance_df.head(15).sort_values(
    "Importance"
).plot(
    x="Feature",
    y="Importance",
    kind="barh",
    figsize=(10, 7),
    legend=False
)

plt.title("Top 15 Feature Importances - Balanced Random Forest")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()
plt.show()



# ==========================================================
# SECTION 33 — EXPERIMENT 5: FEATURE ABLATION
# ==========================================================

# Experiment 5: Train without the two most important features

reduced_numeric_features = [
    col for col in numeric_features
    if col not in ["Hour_Number", "(mph)"]
]

reduced_preprocessor = ColumnTransformer(
    transformers=[
        ("num",
         SimpleImputer(strategy="median"),
         reduced_numeric_features),

        ("cat",
         categorical_transformer,
         categorical_features)
    ]
)

reduced_model = Pipeline(steps=[
    ("preprocessing", reduced_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

reduced_model.fit(X_train_time, y_train_time)

reduced_predictions = reduced_model.predict(X_test_time)

print("FEATURE ABLATION RESULTS")
print("------------------------")
print("Accuracy:",
      accuracy_score(y_test_time, reduced_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test_time,
    reduced_predictions
))



# ==========================================================
# SECTION 34 — FINAL EXPERIMENT COMPARISON
# ==========================================================

final_comparison = pd.DataFrame({
    "Experiment": [
        "Decision Tree",
        "Random Forest",
        "Tuned Random Forest",
        "Chronological Random Forest",
        "Balanced Random Forest",
        "Feature Ablation"
    ],
    "Accuracy": [
        accuracy_score(y_test, dt_predictions),
        accuracy_score(y_test, rf_predictions),
        accuracy_score(y_test, tuned_predictions),
        accuracy_score(y_test_time, time_predictions),
        accuracy_score(y_test_time, balanced_predictions),
        accuracy_score(y_test_time, reduced_predictions)
    ],
    "Macro F1": [
        f1_score(y_test, dt_predictions, average="macro"),
        f1_score(y_test, rf_predictions, average="macro"),
        f1_score(y_test, tuned_predictions, average="macro"),
        f1_score(y_test_time, time_predictions, average="macro"),
        f1_score(y_test_time, balanced_predictions, average="macro"),
        f1_score(y_test_time, reduced_predictions, average="macro")
    ]
})

display(final_comparison.round(4))



# SECTION 35: FINAL PROJECT CONCLUSION
print("\\n" + "=" * 65)
print("FINAL PROJECT CONCLUSION")
print("=" * 65)
print("""
Six model experiments were compared.

The Tuned Random Forest achieved the strongest random-split result:
approximately 90.87% accuracy and 0.91 macro F1-score.

The Balanced Random Forest achieved the stronger chronological result:
approximately 83.65% accuracy and 0.80 macro F1-score. It improved
High Traffic recall compared with the ordinary chronological model,
but High Traffic recall remained a limitation.

Feature ablation showed that removing Hour_Number and (mph) reduced
chronological accuracy substantially. These features provide important
predictive information, although speed may act as a proxy for traffic flow.

The project classifies traffic conditions from available observed
features. It does not establish reliable advance forecasting or
readiness for real-world deployment.

Dataset 2 is the Metro Interstate Traffic Volume dataset from the
UCI Machine Learning Repository. Dataset 1's original source and
licence have not been independently verified and must not be claimed
as verified without the original source documentation.
""")
print("PROJECT PROGRAM FINISHED.")
